<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">《从零构建推理模型》</a>一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录 D：使用更大的 LLM

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # 用于下载函数
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

- 正文各章使用 Qwen3 0.6B 基础模型，因为它是 Qwen3 家族中最小的模型，最容易在消费级硬件上运行
- 不过，附录 C 中的同一个 `Qwen3Model` 实现也可以使用这套从零编写的 PyTorch 模型代码，加载更大的稠密 Qwen3 检查点


&nbsp;
## D.1 更大的稠密 Qwen3 配置

仓库在 `reasoning_from_scratch.appendix_c`（[reasoning_from_scratch/appendix_c.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/appendix_c.py)）中提供了多个比 0.6B 更大的稠密 Qwen3 模型配置字典：

| 模型大小 | 配置字典 |
| --- | --- |
| 1.7B | `QWEN3_CONFIG_1_7B` |
| 4B | `QWEN3_CONFIG_4B` |
| 8B | `QWEN3_CONFIG_8B` |
| 14B | `QWEN3_CONFIG_14B` |
| 32B | `QWEN3_CONFIG_32B` |


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-d/Appendix_D_F01_raschka.webp" width="500px">

- 如上图所述，这些是可在单张 GPU 上运行的“稠密”Qwen3 变体
- Qwen3 还有“稀疏”的混合专家变体，但本书代码不支持；如果对从零实现感兴趣，可参阅：https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/11_qwen3
- 所有这些模型都采用与附录 C 中 0.6B 模型相同的整体架构模式
- 变化的是嵌入大小、层数、注意力头数和前馈网络隐藏维度


- 粗略估算，使用 bfloat16 保存权重时，每个参数约需 2 字节
- 因此，仅检查点权重的内存需求大约为：

| 模型大小 | bfloat16 权重的粗略内存需求 |
| --- | --- |
| 1.7B | 约 3.4 GB |
| 4B | 约 8 GB |
| 8B | 约 16 GB |
| 14B | 约 28 GB |
| 32B | 约 64 GB |


- 实际运行时的内存用量更高，因为还需要为激活值、临时缓冲区以及通常会使用的 KV 缓存分配内存。

&nbsp;
## D.2 下载更大检查点概览

- 与正文各章使用的 0.6B 检查点不同，官方较大 Qwen3 模型通常以 `safetensors` 文件分发，有时会拆分为多个分片
- 使用辅助函数 `download_from_huggingface_from_snapshots` 加载它们需要安装一些额外软件包：


```bash
!uv add huggingface_hub safetensors
```

或者

```bash
!pip install huggingface_hub safetensors
```


&nbsp;
## D.3 加载更大的基础模型

- 下载权重：

In [ ]:
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.appendix_c import (
    download_from_huggingface_from_snapshots
)


device = get_device()
local_dir = Path("qwen3-4b-base")

weights = download_from_huggingface_from_snapshots(
    repo_id="Qwen/Qwen3-4B-Base",
    local_dir=local_dir,
)

- 初始化模型：

In [ ]:
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, load_hf_weights_into_qwen
)
from reasoning_from_scratch.appendix_c import QWEN3_CONFIG_4B


model = Qwen3Model(QWEN3_CONFIG_4B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN3_CONFIG_4B["n_layers"],
        "hidden_dim": QWEN3_CONFIG_4B["hidden_dim"],
    },
    params=weights,
)
model.to(device)
model.eval()

- 加载分词器：

In [ ]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
import shutil

# 请注意，原始基础分词器名为 "tokenizer.json"
# 将其重命名，以区别于下一节的推理分词器
tokenizer_src = local_dir / "tokenizer.json"
tokenizer_path = local_dir / "tokenizer-base.json"

if not tokenizer_path.exists():
    shutil.copyfile(tokenizer_src, tokenizer_path)

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 使用模型：

In [ ]:
import torch
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache,
)

prompt = "Explain large language models in two sentences."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
).unsqueeze(0)

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=64,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

&nbsp;
## D.4 加载更大的推理变体

- 同样的方法也适用于更大的推理风格 Qwen3 模型
- 给定模型大小的架构保持不变，只有检查点和分词器设置不同


例如，要加载 4B 推理变体而不是 4B 基础变体，需要：

- 将仓库 ID 从 `Qwen/Qwen3-4B-Base` 改为 `Qwen/Qwen3-4B`；
- 将 `tokenizer.json` 文件复制为 `tokenizer-reasoning.json`；
- 按如下方式初始化分词器：


```python
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_path,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
```

- 其余模型加载和使用代码保持不变。

&nbsp;
## D.5 实用建议

- 本节没有代码。